# Physics-Informed Head — Publication Figures

Produces publication-style figures comparing the physics-informed feature head (notebook 19)
against the baseline frozen ESM-2 and all structural regularization sweeps (notebooks 12–17).

- **Section A** — Physics head vs baseline: sequence-level classification and residue-level epitope alignment.
- **Section B** — Extended cross-model comparison: physics head added to the notebook-14 sweep chart.
- **Section C** — Learned weight analysis: PhysicsProjection weights vs expected physical signs.

All figures are saved as PNG (140 dpi preview) and PDF (300 dpi print) under `results/paper_figures/`.

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'pyproject.toml').exists():
            return candidate
    raise FileNotFoundError('Could not find repo root containing pyproject.toml')


repo_root = find_repo_root(Path.cwd().resolve())
src_dir   = repo_root / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

rsa_results_dir     = repo_root / 'results' / 'rsa_regularization'
ss3_results_dir     = repo_root / 'results' / 'ss3_regularization'
ss3_hi_results_dir  = repo_root / 'results' / 'ss3_high_lambda'
disorder_results_dir= repo_root / 'results' / 'disorder_regularization'
epitope_results_dir = repo_root / 'results' / 'epitope_regularization'
iedb_results_dir    = repo_root / 'results' / 'iedb_regularization'
physics_results_dir = repo_root / 'results' / 'physics_head'
figures_dir         = repo_root / 'results' / 'paper_figures'
figures_dir.mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid', context='paper')
plt.rcParams.update({
    'figure.dpi':        140,
    'savefig.dpi':       300,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'font.family':       'DejaVu Sans',
})


def _lambda_dir(v: float) -> str:
    return f'lambda_{int(v)}' if float(v).is_integer() else f'lambda_{str(v).replace(".", "p")}'


def _load_sweep(results_dir: Path, lambda_col: str, baseline_row=None) -> tuple:
    """Load sweep_summary.csv + residue CI + loss data."""
    summary = (
        pd.read_csv(results_dir / 'sweep_summary.csv')
        .rename(columns={'lambda_rsa': lambda_col})
        .sort_values(lambda_col).reset_index(drop=True)
    )
    summary['lambda_label'] = summary[lambda_col].map(lambda v: f'{v:g}')
    summary['x']            = np.arange(len(summary))

    if baseline_row is None:
        baseline_row = summary.loc[summary[lambda_col].eq(0.0)].iloc[0]

    for m in ['test_mcc', 'test_auroc', 'test_f1', 'test_accuracy',
              'test_precision', 'test_recall',
              'residue_auroc', 'residue_auprc', 'residue_precision_at_k']:
        summary[f'delta_{m}'] = summary[m] - float(baseline_row[m])

    ci_rows = []
    for row in summary.itertuples(index=False):
        lval = getattr(row, lambda_col)
        mp   = json.loads((results_dir / _lambda_dir(lval) / 'metrics.json').read_text())
        rs   = mp['residue_summary']
        for label, (mk, lk, hk, bl) in {
            'Pooling attention AUROC':        ('auroc_mean',          'auroc_ci_low',          'auroc_ci_high',          float(baseline_row['residue_auroc'])),
            'Pooling attention AUPRC':        ('auprc_mean',          'auprc_ci_low',          'auprc_ci_high',          float(baseline_row['residue_auprc'])),
            'Pooling attention precision@k': ('precision_at_k_mean', 'precision_at_k_ci_low', 'precision_at_k_ci_high', float(baseline_row['residue_precision_at_k'])),
        }.items():
            ci_rows.append({lambda_col: lval, 'x': row.x, 'metric': label,
                            'delta_mean': rs[mk]-bl, 'delta_low': rs[lk]-bl, 'delta_high': rs[hk]-bl})
    return summary, pd.DataFrame(ci_rows)


# ── Load regularization sweeps (same as notebook 14) ──────────────────────────
rsa_summary,     rsa_ci     = _load_sweep(rsa_results_dir, 'lambda_rsa')
rsa_baseline                = rsa_summary.loc[rsa_summary['lambda_rsa'].eq(0.0)].iloc[0]

ss3_lo,  ss3_lo_ci  = _load_sweep(ss3_results_dir,    'lambda_ss3', rsa_baseline)
ss3_hi,  ss3_hi_ci  = _load_sweep(ss3_hi_results_dir, 'lambda_ss3', rsa_baseline)
ss3_summary = pd.concat([ss3_lo, ss3_hi]).sort_values('lambda_ss3').reset_index(drop=True)
ss3_summary['x'] = np.arange(len(ss3_summary))

disorder_summary, _ = _load_sweep(disorder_results_dir, 'lambda_disorder', rsa_baseline)
epitope_summary,  _ = _load_sweep(epitope_results_dir,  'lambda_epitope',  rsa_baseline)
iedb_summary,     _ = _load_sweep(iedb_results_dir,     'lambda_iedb')
iedb_baseline       = iedb_summary.loc[iedb_summary['lambda_iedb'].eq(0.0)].iloc[0]

# ── Load physics head results (notebook 19) ────────────────────────────────────
ph_metrics_path = physics_results_dir / 'metrics.json'
if not ph_metrics_path.exists():
    raise FileNotFoundError(
        f'Physics head metrics not found at {ph_metrics_path}.\n'
        'Run notebook 19 (sections 5–6) first.'
    )
ph_metrics      = json.loads(ph_metrics_path.read_text())
ph_cls          = ph_metrics['classification']
ph_align        = ph_metrics['epitope_alignment']
ph_per_protein  = pd.DataFrame(ph_metrics['per_protein_alignment'])

ph_weight_path  = physics_results_dir / 'weight_summary.csv'
ph_weights      = pd.read_csv(ph_weight_path)

print('Baseline:      test_auroc={:.4f}  test_mcc={:.4f}  residue_auroc={:.4f}'.format(
    float(rsa_baseline['test_auroc']), float(rsa_baseline['test_mcc']),
    float(rsa_baseline['residue_auroc'])))
print('Physics head:  test_auroc={:.4f}  test_mcc={:.4f}  residue_auroc={:.4f}'.format(
    ph_cls['auroc'], ph_cls['mcc'], ph_align['mean_attention_auroc']))
print(f'Physics head Wilcoxon p = {ph_align["wilcoxon_p"]:.2e}')

---
## Section A — Physics Head vs Baseline

Direct comparison of the physics-informed head (notebook 19) against the unregularized frozen ESM-2 baseline
(λ = 0 from the RSA sweep, shared across all structural experiments).

- **Left panel**: sequence-level classification metrics (absolute values) for both models.
- **Right panel**: per-protein pooling-attention AUROC distribution on IEDB splitB (61 proteins).
  The dashed line marks the baseline mean; the significance annotation shows the paired Wilcoxon test
  of the physics head distribution against the random baseline of 0.5.

In [ ]:
DISPLAY_COLS = ['model', 'test_auroc', 'test_f1', 'test_mcc', 'test_accuracy', 'residue_auroc']

comparison_ab = pd.DataFrame([
    {
        'model':          'Baseline (λ=0)',
        'test_auroc':     float(rsa_baseline['test_auroc']),
        'test_f1':        float(rsa_baseline['test_f1']),
        'test_mcc':       float(rsa_baseline['test_mcc']),
        'test_accuracy':  float(rsa_baseline['test_accuracy']),
        'residue_auroc':  float(rsa_baseline['residue_auroc']),
    },
    {
        'model':          'Physics Head',
        'test_auroc':     ph_cls['auroc'],
        'test_f1':        ph_cls['f1'],
        'test_mcc':       ph_cls['mcc'],
        'test_accuracy':  ph_cls['accuracy'],
        'residue_auroc':  ph_align['mean_attention_auroc'],
    },
], columns=DISPLAY_COLS).round(4)
comparison_ab['Δ test_auroc'] = comparison_ab['test_auroc'].diff()
comparison_ab['Δ test_mcc']   = comparison_ab['test_mcc'].diff()
comparison_ab['Δ residue_auroc'] = comparison_ab['residue_auroc'].diff()
comparison_ab

In [ ]:
SEQ_METRICS = [
    ('test_auroc',    'Test AUROC'),
    ('test_f1',       'Test F1'),
    ('test_mcc',      'Test MCC'),
    ('test_accuracy', 'Test accuracy'),
]
COLORS = {'Baseline (λ=0)': '#94a3b8', 'Physics Head': '#0f766e'}
MODELS = ['Baseline (λ=0)', 'Physics Head']
x      = np.arange(len(SEQ_METRICS))
width  = 0.32

p     = ph_align['wilcoxon_p']
stars = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
bl_res_auroc = float(rsa_baseline['residue_auroc'])

fig, axes = plt.subplots(1, 2, figsize=(11.2, 4.2), constrained_layout=True)

# Left: sequence-level metrics (grouped bars)
for i, model in enumerate(MODELS):
    row    = comparison_ab.loc[comparison_ab['model'].eq(model)].iloc[0]
    offset = (i - 0.5) * width
    axes[0].bar(
        x + offset, [float(row[c]) for c, _ in SEQ_METRICS],
        width=width * 0.9, color=COLORS[model], label=model, alpha=0.88,
    )
axes[0].set_xticks(x, [label for _, label in SEQ_METRICS])
axes[0].set_ylabel('Metric value')
axes[0].set_title('Sequence-level classification')
axes[0].legend(frameon=False, fontsize=8)
axes[0].set_ylim(0.75, 1.02)
axes[0].grid(True, axis='y', color='#e2e8f0', linewidth=0.8)
axes[0].grid(False, axis='x')
axes[0].tick_params(axis='x', labelsize=8.5)

# Right: per-protein attention AUROC distribution (physics head)
axes[1].boxplot(
    ph_per_protein['auroc'],
    widths=0.45, patch_artist=True,
    boxprops=dict(facecolor='#0f766e', alpha=0.8),
    medianprops=dict(color='white', linewidth=2),
    whiskerprops=dict(linewidth=1), capprops=dict(linewidth=1),
    flierprops=dict(marker='.', markersize=3, alpha=0.5),
)
axes[1].axhline(bl_res_auroc, color='#94a3b8', linewidth=1.4, linestyle='--',
                label=f'Baseline mean ({bl_res_auroc:.3f})')
axes[1].axhline(0.5, color='#cbd5e1', linewidth=0.9, linestyle=':',
                label='Random (0.5)')

# Significance annotation
ph_aurocs = ph_per_protein['auroc'].values
y_top     = ph_aurocs.max() + (ph_aurocs.max() - ph_aurocs.min()) * 0.06
axes[1].text(1, y_top, f'{stars}  p = {p:.2e}', ha='center', va='bottom', fontsize=8)
axes[1].set_xticks([1], ['Physics Head'])
axes[1].set_ylabel('Per-protein pooling attention AUROC')
axes[1].set_title(f'Residue-level epitope alignment  (n = {ph_align["n_proteins"]})')
axes[1].set_ylim(ph_aurocs.min() - 0.03, y_top + (ph_aurocs.max() - ph_aurocs.min()) * 0.18)
axes[1].legend(frameon=False, fontsize=7.5, loc='lower right')
axes[1].grid(True, axis='y', color='#e2e8f0', linewidth=0.8)
axes[1].grid(False, axis='x')

for png, pdf in [
    (figures_dir / 'main_physics_head_vs_baseline.png',
     figures_dir / 'main_physics_head_vs_baseline.pdf'),
]:
    fig.savefig(png, bbox_inches='tight')
    fig.savefig(pdf, bbox_inches='tight')
plt.show()

**Caption (Section A):** Left: sequence-level classification metrics on the DeepAlgPro test set for the unregularized baseline and the physics-informed head. Right: per-protein pooling-attention AUROC on 61 IEDB splitB allergen proteins. Dashed line shows the baseline mean; dotted line shows the random baseline (0.5). Significance annotation from a one-sided paired Wilcoxon signed-rank test (physics head > 0.5). Fill in interpretation once notebook 19 has been run.

---
## Section B — Extended Cross-Model Comparison

The physics head is added to the notebook-14 cross-sweep comparison.
Each regularization sweep is represented by its best-λ checkpoint (selected by validation MCC);
the physics head appears as a single point without a λ dimension.

- **Left panel**: sequence-level classification at the best operating point.
- **Right panel**: residue-level pooling-attention alignment (AUROC, AUPRC, Precision@k).
  AUPRC and Precision@k are not yet computed for the physics head and shown as absent.

In [ ]:
def _pick_best(summary: pd.DataFrame, lambda_col: str, sweep_name: str) -> dict:
    row = summary.loc[summary['val_mcc'].idxmax()]
    return {
        'sweep':       sweep_name,
        'best_lambda': float(row[lambda_col]),
        'val_mcc':        float(row['val_mcc']),
        'test_mcc':       float(row['test_mcc']),
        'test_auroc':     float(row['test_auroc']),
        'test_f1':        float(row['test_f1']),
        'test_accuracy':  float(row['test_accuracy']),
        'residue_auroc':  float(row['residue_auroc']),
        'residue_auprc':  float(row['residue_auprc']),
        'residue_precision_at_k': float(row['residue_precision_at_k']),
    }


COMP_COLS = ['sweep', 'best_lambda', 'val_mcc', 'test_mcc', 'test_auroc',
             'test_f1', 'test_accuracy', 'residue_auroc', 'residue_auprc',
             'residue_precision_at_k']

comparison_full = pd.DataFrame([
    # Unregularized baseline
    {'sweep': 'Baseline (λ=0)', 'best_lambda': 0.0,
     **{c: float(rsa_baseline[c]) for c in COMP_COLS if c not in ('sweep','best_lambda') and c in rsa_baseline.index}},
    # Regularization best-λ
    _pick_best(rsa_summary.loc[rsa_summary['lambda_rsa'].ne(0.0)],             'lambda_rsa',      'RSA'),
    _pick_best(ss3_summary,                                                      'lambda_ss3',      'SS3-structured'),
    _pick_best(disorder_summary,                                                 'lambda_disorder', 'Disorder'),
    _pick_best(epitope_summary.loc[epitope_summary['lambda_epitope'].ne(0.0)],  'lambda_epitope',  'Epitope'),
    # IEDB sweep (own λ=0 SFT baseline)
    {'sweep': 'IEDB SFT baseline (λ=0)', 'best_lambda': 0.0,
     **{c: float(iedb_baseline[c]) for c in COMP_COLS if c not in ('sweep','best_lambda') and c in iedb_baseline.index}},
    _pick_best(iedb_summary.loc[iedb_summary['lambda_iedb'].ne(0.0)],           'lambda_iedb',     'IEDB (SFT)'),
    # Physics head (no λ sweep; residue AUPRC / Prec@k not computed)
    {'sweep': 'Physics Head', 'best_lambda': float('nan'),
     'val_mcc':       float('nan'),
     'test_mcc':      ph_cls['mcc'],
     'test_auroc':    ph_cls['auroc'],
     'test_f1':       ph_cls['f1'],
     'test_accuracy': ph_cls['accuracy'],
     'residue_auroc': ph_align['mean_attention_auroc'],
     'residue_auprc': float('nan'),
     'residue_precision_at_k': float('nan')},
], columns=COMP_COLS).round(4)

comparison_full

In [ ]:
SWEEP_COLORS = {
    'Baseline (λ=0)':          '#94a3b8',
    'RSA':                     '#0f766e',
    'SS3-structured':          '#7c3aed',
    'Disorder':                '#ea580c',
    'Epitope':                 '#0891b2',
    'IEDB SFT baseline (λ=0)': '#6b7280',
    'IEDB (SFT)':              '#16a34a',
    'Physics Head':            '#b45309',
}
SWEEPS = list(SWEEP_COLORS)

SEQ_M = [('test_mcc',      'Test MCC'),
         ('test_auroc',    'Test AUROC'),
         ('test_f1',       'Test F1'),
         ('test_accuracy', 'Test accuracy')]
RES_M = [('residue_auroc',            'Pooling\nAUROC'),
         ('residue_auprc',            'Pooling\nAUPRC'),
         ('residue_precision_at_k',   'Pooling\nPrec@k')]

x_s = np.arange(len(SEQ_M))
x_r = np.arange(len(RES_M))
w   = 0.095

fig, axes = plt.subplots(1, 2, figsize=(15, 4.4), constrained_layout=True)

for i, sweep_name in enumerate(SWEEPS):
    rows = comparison_full.loc[comparison_full['sweep'].eq(sweep_name)]
    if rows.empty:
        continue
    row    = rows.iloc[0]
    color  = SWEEP_COLORS[sweep_name]
    offset = (i - (len(SWEEPS) - 1) / 2) * w

    axes[0].bar(
        x_s + offset, [float(row[c]) for c, _ in SEQ_M],
        width=w * 0.88, color=color, label=sweep_name, alpha=0.88,
    )
    res_vals = []
    for c, _ in RES_M:
        v = float(row[c])
        res_vals.append(v if not np.isnan(v) else 0.0)
    # Use hatch to mark missing values (physics head: no AUPRC / Prec@k)
    hatches = ['' if not np.isnan(float(row[c])) else '///' for c, _ in RES_M]
    for xi, (rv, hatch) in enumerate(zip(res_vals, hatches)):
        axes[1].bar(
            x_r[xi] + offset, rv,
            width=w * 0.88, color=color, alpha=0.88 if not hatch else 0.35,
            hatch=hatch,
        )

axes[0].set_xticks(x_s, [label for _, label in SEQ_M])
axes[0].set_ylabel('Metric value')
axes[0].set_title('Sequence-level classification (best λ per sweep)')
axes[0].legend(frameon=False, fontsize=7, ncol=2)
axes[0].set_ylim(0.75, 1.0)

axes[1].set_xticks(x_r, [label for _, label in RES_M])
axes[1].set_ylabel('Metric value')
axes[1].set_title('Residue-level alignment (best λ per sweep; /// = not computed)')
axes[1].set_ylim(0.0, 0.65)

for ax in axes:
    ax.grid(True, axis='y', color='#e2e8f0', linewidth=0.8)
    ax.grid(False, axis='x')
    ax.tick_params(axis='x', labelsize=8.5)

for ext in ('.png', '.pdf'):
    fig.savefig(figures_dir / f'main_cross_model_comparison_with_physics{ext}', bbox_inches='tight')
plt.show()

**Caption (Section B):** Best-λ comparison of all structural regularization strategies plus the physics-informed head against the unregularized baseline. Sequence-level classification (left) is stable across conditions. Residue-level pooling-attention alignment (right) shows whether any approach shifts attention toward IEDB epitope residues. Hatching (///) indicates metrics not computed for the physics head; these can be added by extending `evaluate.py`. Fill in interpretation once notebook 19 has been run.

---
## Section C — Learned Weight Analysis

The 10 scalar weights of `PhysicsProjection` form a learned linear free-energy equation.
Triangles mark the expected sign from biophysical priors;
green bars indicate the model learnt that channel in a physically plausible direction,
red bars indicate a flipped sign.

In [ ]:
from matplotlib.patches import Patch
from xallergen.physics_head.model import CHANNEL_NAMES, EXPECTED_SIGNS

w_vals    = ph_weights['learned_weight'].astype(float).to_numpy()
preserved = ph_weights['sign_preserved'].tolist()
bar_clr   = ['#0f766e' if p else '#dc2626' for p in preserved]
marker_x  = max(float(np.abs(w_vals).max()) * 0.18, 0.005)

n_dir = sum(1 for es in EXPECTED_SIGNS if es != 0)
n_ok  = sum(1 for p, es in zip(preserved, EXPECTED_SIGNS) if es != 0 and p)

fig, ax = plt.subplots(figsize=(7.5, 4.5), constrained_layout=True)

ax.barh(range(10), w_vals, color=bar_clr, alpha=0.85, edgecolor='none')
ax.axvline(0, color='k', linewidth=0.8)
for i, es in enumerate(EXPECTED_SIGNS):
    if es != 0:
        ax.plot(es * marker_x, i, f'k{"^" if es > 0 else "v"}',
                markersize=6, alpha=0.65, clip_on=False)

ax.set_yticks(range(10))
ax.set_yticklabels(CHANNEL_NAMES, fontsize=9)
ax.set_xlabel('Learned weight')
ax.set_title(
    f'PhysicsProjection learned weights  '
    f'(sign preserved: {n_ok}/{n_dir} directional channels)'
)
ax.legend(
    handles=[
        Patch(color='#0f766e', label='Sign preserved'),
        Patch(color='#dc2626', label='Sign flipped'),
    ],
    frameon=False, fontsize=8,
)
ax.grid(True, axis='x', color='#e2e8f0', linewidth=0.8)
ax.grid(False, axis='y')

for ext in ('.png', '.pdf'):
    fig.savefig(figures_dir / f'main_physics_head_weights{ext}', bbox_inches='tight')
plt.show()

**Caption (Section C):** Learned `PhysicsProjection` weights after training. Triangles (▲/▼) show the expected sign from binding-energy priors: RSA, charge, charge×RSA, HB count, and HB×RSA expected positive; hydrophobicity, burial-weighted hydrophobicity, and disorder expected negative; sin/cos φ have no directional prior. Green bars indicate physically plausible learning; red bars indicate flipped signs, which would extend the misalignment finding from residue positions to physicochemical feature use. Fill in interpretation once notebook 19 has been run.